# Week 7: recurring subscription spend per account-month

**Checkpoint (Lecture 7):** one account-month feature from our Q1 category (Subscriptions & Entertainment), with
its transaction filter, amount convention, denominator, zero share, median / IQR / upper tail, a chart, one
surprising pattern and one caveat, keeping **zero activity** separate from **missing coverage**.

Mentor's suggestion for Group 3: *"Subscriptions: recurring spend + active months"* (streaming, software,
recurring gym plans), and feedback to separate recurring services from one-off events and to audit the
gambling and gym boundaries.

Reuses `full_histories_5000accts.parquet` written by `income_feature_5000.ipynb` (full histories of the 513
accounts in our 5,000-memo sample). If that file isn't next to this notebook, section 2 rebuilds it (slow).

In [ ]:
import glob, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import pyarrow.dataset as ds

BASE = "/home/ailai/teams/d12/UCSD - Data Share"
SAMPLE_CSV = "memos_5000.csv"
HIST_CACHE = Path("full_histories_5000accts.parquet")
OUT = Path("outputs_subscriptions_feature"); OUT.mkdir(exist_ok=True)
FAKE = 207405
DEBIT_CODE = 2          # credit_or_debit: 1 = credit, 2 = debit (checked in the income notebook)
MIN_MONTHS = 3          # a merchant is "recurring" for an account if charged in >= 3 distinct months
COLS = ["account_key", "posted_date", "amount", "credit_or_debit", "memo"]

## 1. Category rules

Our week 5 category rules (`label_rules.py`, first match wins), then a **subtype** split inside the category so
we can make the scope call the mentor asked for. Gym plans live under Health & Wellness in our guide, so they
are pulled in here as a subscription subtype.

| subtype | in the feature? | why |
|---|---|---|
| streaming | yes | recurring media services |
| software / apps | yes | app stores, cloud, software plans |
| gym | yes | recurring membership |
| other subscription | yes | remaining Subscriptions & Entertainment memos |
| events / recreation | **no** | one-off purchases (tickets, bowling, venues) |
| gambling | **no** | wagers, not subscriptions; tagged in our guide but excluded here |

In [ ]:
# ---- week 5 category rules (copied from label_rules.py) ----
GAMBLING = (r"draftkings|fanduel|fd sptsbk|fanatics sp|sportsbo|prizepicks|underdog|lottery|valottery"
            r"|\bolg\b|betmgm|hard rock bet|crowncoins|chumba|wowvegas|fliff|sidepot|casino"
            r"|nflallday|meetdapper|powerplay|jbox game|baca game")

# (category, regex, subcategory) in priority order
RULES = [
    # --- specific merchants whose memo also contains a misleading word (fee, bar, paypal ...)
    ("Bills & Utilities", r"agi\*tmo|magicja", None),
    ("Health & Wellness", r"iclub fees|planet fitness|nail bar|hair design", None),
    ("Automotive", r"dmv", None),
    ("Shopping", r"stevenmadde", None),
    ("Uncertain - Ambiguous Merchant", r"get go", None),

    # --- bank fees win over whatever merchant they mention
    ("Fees & Adjustments", r"\bfee\b|fee-|fees\b|overdraft|\bnsf\b|returned item|service charge|over limit"
                           r"|atm rebate|courtesy pay|uncollected protection|return item|reverse check", None),

    # --- gambling (guide has no gambling class -> Subscriptions & Entertainment, tagged)
    ("Subscriptions & Entertainment", GAMBLING, "gambling"),

    # --- cash-advance apps, BNPL, loans, card payments
    ("Transfers", r"dave\.com p2p", None),
    ("Debt / Credit Payments",
     r"\bdave\b|davesubfee|brigit|earnin|moneylion|instacash|cleo|empower|\btilt\b|klover|floatme|float me"
     r"|mypay|grant cash|grant repayment|credit ?genie|oneblinc|beem app|\bvola\b|atlasfinancial"
     r"|advance financia|true fin|flex finance|creditninja|onemain|avant llc|capital one|credit one"
     r"|afterpay|affirm|klarna|quadpay|\bzip\b|sezzle|deferit|progressive leas|progressivelease"
     r"|rent a center|mortgage|\bgrid\b|getgrid|pago de tarjeta|deposit advance", None),

    # --- P2P apps / own-account sweeps (beat the person or business on the other end)
    ("Transfers", r"zelle|\bzel to\b|venmo|cash\s*/?\s*app|apple cash|visa direc|visa money transfer|third prty"
                  r"|e-transfer|moneygram|round ?up|keep the change|save as you go|save your change"
                  r"|albert savings|inst xfer", None),

    # --- early income (memos that also contain the word "transfer"/"deposit")
    ("Income / Deposits", r"payroll|province of b\.c|child benefit|treas 310|\bssi\b|gusto pay|stripe"
                          r"|uber instantpay|uber pro card|gopuff|ccmsi|universal city d batch"
                          r"|interest earned|dividend|fidelity investm", None),

    # --- merchant categories
    ("Subscriptions & Entertainment",
     r"netflix|spotify|hulu|disney|peacock|paramount|prime video|amazon prime|apple\.com/bill|^apple$"
     r"|google|microsoft|audiobooks|audible|roku|philo|tiktok|patreon|chaturbill|of london|abcmouse"
     r"|riot fest|round1|\bbowl\b|stubhub|treehouse indoor|amzn apps|arlo tech|rocket money|creditreview"
     r"|hotmart|candybar", None),
    ("Food & Drink",
     r"mcdonald|burger|chipotle|starbucks|doordash|\bdd \*|uber \*?eats|chick-fil-a|dunkin|popeye"
     r"|church'?s chicken|zaxby|^sonic$|jack in the box|raising cane|waffle house|jersey mike|taco"
     r"|restaura|\bcafe\b|pizza|chicken|\bgrill|\bbar\b|\bpint\b|ice crea|bakery|dining|tst\*|aramark"
     r"|canteen|vending|\bvend\b|payrange|365 market|market@work|seaworld-food|food svc|halal food"
     r"|filibertos|mariscos|pollo campero|tim hortons|cookie|snack shop|cpa clb conc|lounge"
     r"|liquor|liqu\b|\bwine\b|charlie's|nslc", None),
    ("Groceries",
     r"walmart|wal-mart|wm supercenter|safeway|food4less|food lion|kroger|publix|\baldi\b|\blidl\b"
     r"|\bgiant\b|harris te|trader joe|whole foods|ralphs|costco|bjs wholesale|key foods|grocer"
     r"|instacart|maplebear|mariano|westtown foo|carniceria|danny's super f|nesters market|pete's fr", None),
    ("Health & Wellness",
     r"cvs|walgreens|rite aid|pharm|urgent care|dental|la fitness|central florida ym|nail bar"
     r"|hair design|ucsdh|medical|clinic|blue cross", None),
    ("Shopping",
     r"amazon|amzn|etsy|target|aliexpress|tjmaxx|ross store|burlington|winners|jcpenney|five belo"
     r"|dollar general|family dollar|dollar ?tree|clearance outlet|platos closet|the closet|galaxy comics"
     r"|beauty|smokers paradise|vape|exotic shop|lowe's|home depot|stevenmadde|shipmentfree"
     r"|new wave promos|farmersdog|pig & cat|carpe\b", None),
    ("Bills & Utilities",
     r"t-mobile|verizon|at&t|att\*|spectrum|comcast|frontier comm|boost mobile|cable svc|reliant energy"
     r"|georgia power|scana energy|electric|utility|insurance|\bins\b|geico|allstate|state farm"
     r"|natl gen ins|chubb|magicja|laundry|textm", None),
    ("Travel",
     r"uber|ubr\*|lyft|airbnb|marriott|hilton|hotel|meridien|homewood suites|jetblue|airline|greyhound"
     r"|masabi|parking|parkup|toll|e-z\*pass|rental car", None),
    ("Automotive",
     r"car ?wash|laser wash|autozone|o'reilly|advance auto|tire|motors|dmv|auto parts", None),
    ("Professional Services",
     r"law offices|legalnature|turbotax|intuit|accounting|notary|numbers team|got-ju|mowsnowpros", None),
    ("Finance & Investments", r"coinbase|crypto|robinhood|webull|acorns|stash", None),
    ("Gifts & Donations", r"gofundme|donation|world vision|scouting", None),
    ("Education", r"college|university|tuition|school", None),
    ("Gas", r"shell|chevron|exxon|mobil\b|\barco\b|\bbp\b|bp#|citgo|sunoco|amoco|pemex|am gas", None),

    # --- known store, unknown basket: gas-station convenience stores, corner markets, processors
    ("Uncertain - Ambiguous Merchant",
     r"7-?eleven|wawa|racetrac|\bqt\b|quiktrip|kwik trip|get go|enmarket|circle k|speedway|sheetz"
     r"|pic n pump|travel center|^parkers$|hollywood mini|market|\bmart\b|\bconv\b|pantry|\bsq \*|payper|c-rewards"
     r"|ilixium|flexepayments|or technologies|baytree", None),

    # --- generic money movement after all merchant checks
    ("Transfers", r"transfer|\btfr|xfer|from savings|to shares?\b|credit builder|savings|checking"
                  r"|way2save|from wallet", None),
    ("Income / Deposits", r"deposit", None),
    ("Cash Out (ATM)", r"\batm\b|cash withdrawal|withdrwl", None),
    ("Fees & Adjustments", r"return|refund|adjustment|correction", None),
]

_COMPILED = [(cat, re.compile(pat), sub) for cat, pat, sub in RULES]

def suggest_label(text_lower):
    for cat, rx, sub in _COMPILED:
        m = rx.search(text_lower)
        if m:
            return cat, m.group(0), sub
    return "Uncertain - Unidentifiable", "", None

# ---- subtypes inside the category (checked in this order) ----
SUBTYPES = [
    ("gambling", GAMBLING),
    ("gym", r"planet fitness|la fitness|central florida ym|\bymca\b|iclub|fitness|\bgym\b|crunch|anytime fit|orangetheory"),
    ("events / recreation", r"riot fest|round1|\bbowl|stubhub|ticketmaster|treehouse indoor|cinemark|\bamc\b|regal"
                            r"|theat|candybar|seaworld|six flags|dave & buster|topgolf|eventbrite"),
    ("streaming", r"netflix|spotify|hulu|disney|peacock|paramount|prime video|amazon prime|roku|philo|youtube"
                  r"|hbo|\bmax\b|sling|siriusxm|pandora|audible|audiobooks|crunchyroll|fubo"),
    ("software / apps", r"apple\.com/bill|^apple$|google|microsoft|amzn apps|patreon|tiktok|rocket money|creditreview"
                        r"|arlo tech|abcmouse|hotmart|adobe|dropbox|icloud|openai|chatgpt|xbox|playstation|nintendo"),
]
_SUB_RX = [(name, re.compile(p)) for name, p in SUBTYPES]
IN_FEATURE = {"streaming", "software / apps", "gym", "other subscription"}

def lower_text(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()

def classify(text):
    # returns (category, subtype, merchant_key); subtype None if not in our slice
    cat, cue, _ = suggest_label(text)
    for name, rx in _SUB_RX:
        m = rx.search(text)
        if m and (cat == "Subscriptions & Entertainment" or (name == "gym" and cat == "Health & Wellness")):
            return cat, name, m.group(0)
    if cat == "Subscriptions & Entertainment":
        return cat, "other subscription", cue
    return cat, None, None

## 2. Load full histories for the sampled accounts

In [ ]:
if HIST_CACHE.exists():
    hist = pd.read_parquet(HIST_CACHE)
    print("loaded cache:", len(hist), "rows")
else:
    sample = pd.read_csv(SAMPLE_CSV, dtype={"account_key": "Int64"})
    accounts = sorted(sample["account_key"].dropna().astype(int).unique().tolist())
    files = sorted(glob.glob(f"{BASE}/ucsd_transactions_part_*.parquet"))
    flt = ds.field("account_key").isin(accounts)
    parts = []
    for k, f in enumerate(files, 1):
        parts.append(ds.dataset(f, format="parquet").to_table(columns=COLS, filter=flt).to_pandas())
        print(f"[{k}/{len(files)}] {sum(map(len, parts)):,} rows so far")
    hist = pd.concat(parts, ignore_index=True)
    hist["memo"] = hist["memo"].map(lambda b: b.decode("utf-8", "replace") if isinstance(b, bytes) else b)
    hist.to_parquet(HIST_CACHE, index=False)

hist = hist[hist["account_key"] != FAKE].copy()
hist["amount"] = hist["amount"].astype(float)   # parquet stores Decimal
hist["date"] = pd.to_datetime(hist["posted_date"])
hist["month"] = hist["date"].dt.to_period("M")
hist["is_debit"] = hist["credit_or_debit"] == DEBIT_CODE
print(f"{len(hist):,} transactions, {hist.account_key.nunique()} accounts, "
      f"{hist.date.min().date()} to {hist.date.max().date()}, {hist.is_debit.sum():,} debits")

## 3. Label every debit (unique memos only, so it stays fast)

In [ ]:
deb = hist[hist["is_debit"]].copy()
deb["text"] = deb["memo"].map(lower_text)
uniq = deb["text"].unique()
print(f"{len(uniq):,} distinct debit memos to label ...")
lab = {t: classify(t) for t in uniq}
deb["category"] = deb["text"].map(lambda t: lab[t][0])
deb["subtype"] = deb["text"].map(lambda t: lab[t][1])
deb["merchant"] = deb["text"].map(lambda t: lab[t][2])
slice_ = deb[deb["subtype"].notna()].copy()
print(f"{len(slice_):,} debits in Subscriptions & Entertainment (+ gym)")

## 4. Scope check: what is actually inside the category?

This is the boundary audit the mentor asked for. Gambling and one-off events are counted but kept out of the
feature.

In [ ]:
by_sub = (slice_.groupby("subtype")
          .agg(rows=("amount", "size"), dollars=("amount", "sum"), accounts=("account_key", "nunique"))
          .assign(dollar_share=lambda t: t.dollars / t.dollars.sum(),
                  in_feature=lambda t: t.index.isin(IN_FEATURE))
          .sort_values("dollars", ascending=False))
display(by_sub.round(3))

top_merch = (slice_.groupby(["subtype", "merchant"])
             .agg(rows=("amount", "size"), dollars=("amount", "sum"), accounts=("account_key", "nunique"))
             .sort_values("dollars", ascending=False))
display(top_merch.head(25).round(0))

# examples for the label guide (blank human_label for the group to correct)
review = (slice_.groupby(["subtype", "merchant", "memo"])
          .agg(rows=("amount", "size"), dollars=("amount", "sum")).reset_index()
          .sort_values(["subtype", "rows"], ascending=[True, False])
          .groupby("subtype").head(40))
review["human_label"] = ""
review.to_csv(OUT / "subscription_examples_review.csv", index=False)
print(len(review), "example memos ->", OUT / "subscription_examples_review.csv")

## 5. Recurrence: which subscriptions actually recur?

A subscription merchant counts as **recurring** for an account if that account is charged by it in at least
`MIN_MONTHS` (= 3) distinct calendar months. The merchant is the keyword the rule matched (e.g. `netflix`,
`spotify`, `planet fitness`), so the same service is grouped even when the memo text varies.

In [ ]:
subs = slice_[slice_["subtype"].isin(IN_FEATURE)].copy()
am_merch = subs.groupby(["account_key", "merchant"])["month"].nunique().rename("months_charged").reset_index()
am_merch["recurring"] = am_merch["months_charged"] >= MIN_MONTHS
subs = subs.merge(am_merch[["account_key", "merchant", "recurring"]], on=["account_key", "merchant"])

rec_share = subs.groupby("recurring")["amount"].sum() / subs["amount"].sum()
print(f"subscription dollars that are recurring: {rec_share.get(True, 0):.1%}")
display(subs.groupby(["subtype", "recurring"])["amount"].agg(rows="size", dollars="sum").round(0))

## 6. Feature definition + account-month grid

**Feature: `recurring_sub_spend` = monthly recurring subscription dollars per account-month.**

- **Transaction filter:** debits (`credit_or_debit == 2`) labeled Subscriptions & Entertainment by our rules (plus
  gym memberships), in the streaming, software / apps, gym or other-subscription subtypes, from a merchant the
  account is charged by in at least 3 distinct months. Gambling and events / recreation are excluded.
- **Amount convention:** sum of the positive debit `amount`, in dollars, by calendar month of `posted_date`.
  Refunds are not netted.
- **Second measure (active months):** per account, the share of its covered months with
  `recurring_sub_spend > 0`.

Coverage status, as in the income notebook: `covered` (full month with any transaction: a $0 is a real zero),
`no_data_gap` (no transactions at all inside the account's span: missing coverage), `partial_edge` (first or
last month: only partly observed). Only `covered` months enter the denominator.

In [ ]:
rec = subs[subs["recurring"]]
feat = rec.groupby(["account_key", "month"]).agg(recurring_sub_spend=("amount", "sum"),
                                                 n_charges=("amount", "size"),
                                                 n_services=("merchant", "nunique")).reset_index()
act = hist.groupby(["account_key", "month"]).size().rename("n_txn").reset_index()

span = hist.groupby("account_key")["month"].agg(first="min", last="max").reset_index()
grid = pd.concat([pd.DataFrame({"account_key": r.account_key,
                                "month": pd.period_range(r.first, r.last, freq="M")})
                  for r in span.itertuples()], ignore_index=True)
am = (grid.merge(span, on="account_key")
          .merge(act, on=["account_key", "month"], how="left")
          .merge(feat, on=["account_key", "month"], how="left"))
am["n_txn"] = am["n_txn"].fillna(0).astype(int)
for c in ["recurring_sub_spend", "n_charges", "n_services"]:
    am[c] = am[c].fillna(0)
am["status"] = np.select([(am.month == am["first"]) | (am.month == am["last"]), am.n_txn == 0],
                         ["partial_edge", "no_data_gap"], "covered")
am = am.drop(columns=["first", "last"])
display(am["status"].value_counts().rename("account_months").to_frame())

## 7. Distribution over covered account-months

In [ ]:
cov = am[am["status"] == "covered"]
x = cov["recurring_sub_spend"]
nz = x[x > 0]
q = x.quantile([.25, .5, .75, .9, .95, .99])
per_acct = cov.groupby("account_key").agg(months=("month", "size"),
                                          active_months=("recurring_sub_spend", lambda s: int((s > 0).sum())))
per_acct["active_share"] = per_acct.active_months / per_acct.months
summary = {
    "denominator_account_months": int(len(cov)),
    "accounts": int(cov.account_key.nunique()),
    "zero_share": float((x == 0).mean()),
    "median": float(q[.5]), "p25": float(q[.25]), "p75": float(q[.75]), "iqr": float(q[.75] - q[.25]),
    "p90": float(q[.9]), "p95": float(q[.95]), "p99": float(q[.99]), "max": float(x.max()),
    "nonzero_median": float(nz.median()) if len(nz) else None,
    "nonzero_p25": float(nz.quantile(.25)) if len(nz) else None,
    "nonzero_p75": float(nz.quantile(.75)) if len(nz) else None,
    "nonzero_p95": float(nz.quantile(.95)) if len(nz) else None,
    "excluded_no_data_gap": int((am.status == "no_data_gap").sum()),
    "excluded_partial_edge": int((am.status == "partial_edge").sum()),
    "accounts_never_active": int((per_acct.active_months == 0).sum()),
    "accounts_active_every_month": int((per_acct.active_share == 1).sum()),
    "median_active_share": float(per_acct.active_share.median()),
    "recurring_share_of_subscription_dollars": float(rec_share.get(True, 0)),
    "gambling_share_of_category_dollars": float(by_sub.dollar_share.get("gambling", 0)),
}
pd.Series(summary).to_frame("value")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), gridspec_kw={"width_ratios": [1, 2, 1.2]})

parts = pd.Series({
    "spend\n> 0": int((x > 0).sum()),
    "spend\n= 0": int((x == 0).sum()),
    "missing\n(no data)": summary["excluded_no_data_gap"],
    "partial\nmonth": summary["excluded_partial_edge"],
})
axes[0].bar(parts.index, parts.values, color=["#2a7fba", "#9cc3e0", "#c9c9c9", "#e2e2e2"])
axes[0].set_ylabel("account-months")
axes[0].set_title("Zero spend vs missing coverage")
axes[0].set_ylim(0, parts.max() * 1.12)
for i, v in enumerate(parts.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")

if len(nz):
    bins = np.logspace(np.log10(max(nz.min(), 1)), np.log10(nz.max()), 35)
    axes[1].hist(nz, bins=bins, color="#9cc3e0", edgecolor="white")
    axes[1].set_xscale("log")
    for name, v, col, ls in [("median", nz.median(), "#d62728", "-"),
                             ("p75", nz.quantile(.75), "#ff7f0e", "--"),
                             ("p95", nz.quantile(.95), "#2ca02c", ":")]:
        axes[1].axvline(v, color=col, ls=ls, lw=2.2, label=f"{name}: ${v:,.0f}")
    axes[1].legend(title="spend > 0", loc="upper right", fontsize=11, title_fontsize=11, frameon=True)
    axes[1].xaxis.set_major_formatter(FuncFormatter(
        lambda v, _: f"${v/1000:,.0f}k" if v >= 1000 else f"${v:,.0f}"))
axes[1].set_xlabel("monthly recurring subscription dollars (log scale)")
axes[1].set_ylabel("covered account-months")
axes[1].set_title(f"Non-zero recurring_sub_spend ({summary['zero_share']:.0%} of months are $0, not shown)")

axes[2].hist(per_acct.active_share, bins=np.linspace(0, 1, 11), color="#2a7fba", edgecolor="white")
axes[2].set_xlabel("share of covered months with a recurring charge")
axes[2].set_ylabel("accounts")
axes[2].set_title("Active months per account")

plt.tight_layout()
plt.savefig(OUT / "recurring_sub_spend_distribution.png", dpi=150)
plt.show()

## 8. Material for the surprising pattern and the caveat

In [ ]:
# (a) what dominates the upper tail?
top = cov.nlargest(10, "recurring_sub_spend")[["account_key", "month", "recurring_sub_spend", "n_charges", "n_services"]]
display(top)
keys = set(zip(top.account_key, top.month.astype(str)))
in_top = pd.Series([(a, str(m)) in keys for a, m in zip(rec.account_key, rec.month)], index=rec.index)
display(rec[in_top].groupby(["subtype", "merchant"])["amount"].agg(["size", "sum"])
        .sort_values("sum", ascending=False).head(15).round(0))

# (b) most common recurring services across accounts
display(rec.groupby("merchant").agg(accounts=("account_key", "nunique"), dollars=("amount", "sum"),
                                    median_charge=("amount", "median"))
           .sort_values("accounts", ascending=False).head(15).round(2))

# (c) label caveat: catch-all "other subscription" and broad cues (google / apple / amazon) can be non-subscriptions
broad = subs[subs["merchant"].isin(["google", "apple.com/bill", "apple", "microsoft"])]
print(f"subscription dollars from broad app-store / platform cues: {broad.amount.sum() / subs.amount.sum():.1%}")
print(f"subscription dollars in the catch-all 'other subscription' subtype: "
      f"{subs.loc[subs.subtype == 'other subscription', 'amount'].sum() / subs.amount.sum():.1%}")

In [ ]:
am.assign(month=am.month.astype(str)).to_csv(OUT / "account_month_subscriptions.csv", index=False)
per_acct.to_csv(OUT / "account_active_months.csv")
(OUT / "subscriptions_feature_summary.json").write_text(json.dumps(summary, indent=2))
print(sorted(p.name for p in OUT.iterdir()))

## 9. Write-up (fill in from the numbers above)

- **Feature formula:** `recurring_sub_spend` = sum of debit dollars per account-month from streaming,
  software / apps, gym and other subscription merchants that the account is charged by in at least 3 distinct
  months. Gambling and one-off events excluded.
- **Account-month chart:** `outputs_subscriptions_feature/recurring_sub_spend_distribution.png`
- **Denominator:** ___ covered account-months from ___ accounts (excluded: ___ no-data months, ___ partial months).
- **Distribution:** zero share ___%; median $___; IQR $___ to $___; p95 $___; p99 $___.
- **Active months:** median account has a recurring charge in ___% of its covered months; ___ accounts never do.
- **Surprising pattern:** ___
- **Caveat:** ___